# Keyword Extraction

Extract unique lemmatized words (with POS type) from Original, AI-Curated, and Human-Curated phrases using spaCy.

### Input
- `figures/term_mapping.csv` — produced by PhraseAnalysis.ipynb Step 6

### Output
- `figures/unique_words.csv` — 6 columns: (OriginalWord, OriginalPOS), (AIcuratedWord, AIcuratedPOS), (HumanCuratedWord, HumanCuratedPOS)

### Execution guide

**Required:** cells 2-9. They load the phrase mapping, join the source image data, and write the word tables by overall topic and VisType. There are no separate optional analysis cells in this notebook.

In [ ]:
# --- portable setup (added by clean-up) ---
import os, sys
REPO_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
os.makedirs('data_processing', exist_ok=True)
from data_loader import load_sheet, load_paired_comparison
import config


In [ ]:
import pandas as pd
import spacy

nlp = spacy.load('en_core_web_sm')
df = pd.read_csv('data_processing/term_mapping.csv')

def extract_unique_words(phrases):
    """Extract unique (lemma, POS) pairs from a list of phrases, removing stopwords and punctuation."""
    word_pos = set()
    for phrase in phrases:
        if not isinstance(phrase, str) or phrase.strip() in ('', '-'):
            continue
        doc = nlp(phrase)
        for token in doc:
            if token.is_stop or token.is_punct or token.is_space:
                continue
            lemma = token.lemma_.lower()
            if lemma:
                word_pos.add((lemma, token.pos_))
    return sorted(word_pos)

# Extract unique words for each type
cols = {'original': 'Original', 'AIcurated': 'AIcurated', 'HumanCurated': 'HumanCurated'}
word_lists = {}
for col, label in cols.items():
    phrases = df[col][df[col] != ''].unique()
    wp = extract_unique_words(phrases)
    word_lists[label] = wp
    print(f'{label}: {len(phrases)} phrases → {len(wp)} unique words')

# Build output DataFrame — 3 independent column-pairs, padded
max_len = max(len(v) for v in word_lists.values())
frames = []
for label in cols.values():
    wp = word_lists[label]
    d = pd.DataFrame(wp, columns=[f'{label}Word', f'{label}POS'])
    d = d.reindex(range(max_len)).fillna('')
    frames.append(d)

df_out = pd.concat(frames, axis=1)

csv_path = 'data_processing/unique_words.csv'
df_out.to_csv(csv_path, index=False)
print(f'\nSaved {csv_path}  ({max_len} rows)')
df_out.head(15)

In [ ]:
# ── Helper functions ────────────────────────────────────────────────────────────
from collections import Counter

topic_cols = [
    'Immediacy / Cognitive Load',
    'Data Density / Image Clutter',
    'Visual Encoding Clarity',
    'Semantics / Text Legibility',
    'Schema',
    'Color, Symbol, and Texture Details',
    'Aesthetics Uncertainty',
]

def extract_lemmas_from_phrase(phrase):
    """Extract set of lemmatized words from a single phrase."""
    if not isinstance(phrase, str) or phrase.strip() in ('', '-'):
        return set()
    doc = nlp(phrase)
    return {token.lemma_.lower() for token in doc
            if not token.is_stop and not token.is_punct and not token.is_space and token.lemma_.lower()}

def extract_unique_lemmas(phrases):
    """Extract sorted unique lemmatized words from a list of phrases."""
    words = set()
    for phrase in phrases:
        words |= extract_lemmas_from_phrase(phrase)
    return sorted(words)

print('Helper functions defined.')

In [ ]:
# ── Load paired image data (sheet 1103808983) ─────────────────────────────────
import json

spreadsheet_id = '1gfeYdT-RxLq9tvNpfv_P8AIKg2eSz6-8bSv5zUJsWEc'
sheet_id = '1103808983'
df_pairs = load_sheet(spreadsheet_id, sheet_id)
print(f'Shape: {df_pairs.shape}')
print(f'Columns: {df_pairs.columns.tolist()}')

# Filter out rows with ToRemove
before = len(df_pairs)
df_pairs = df_pairs[df_pairs['ToRemove'].isna() | (df_pairs['ToRemove'].astype(str).str.strip() == '')]
print(f'Rows after removing ToRemove: {len(df_pairs)} (removed {before - len(df_pairs)})')
df_pairs.head(3)

In [ ]:
# ── Build (topic, original) → (AIcurated, HumanCurated) mapping from sheet 1166121277 ──
sheet_map_id = '1166121277'
df_map_src = load_sheet(spreadsheet_id, sheet_map_id)

topic_cols = [
    'Immediacy / Cognitive Load',
    'Data Density / Image Clutter',
    'Visual Encoding Clarity',
    'Semantics / Text Legibility',
    'Schema',
    'Color, Symbol, and Texture Details',
    'Aesthetics Uncertainty',
]

# Build topic-aware mapping: (topic, original) → (AIcurated, HumanCurated)
topic_orig_to_curated = {}
for topic in topic_cols:
    col_orig = topic
    col_ai   = f'{topic} (AIcurated)'
    col_hc   = f'{topic} (HumanCuration)'
    for _, row in df_map_src.iterrows():
        orig = row.get(col_orig)
        if not isinstance(orig, str) or orig.strip() == '':
            continue
        orig = orig.strip()
        key = (topic, orig)
        if key not in topic_orig_to_curated:
            ai = str(row.get(col_ai, '')).strip() if isinstance(row.get(col_ai), str) else ''
            hc = str(row.get(col_hc, '')).strip() if isinstance(row.get(col_hc), str) else ''
            topic_orig_to_curated[key] = (ai, hc)

print(f'Mapping entries: {len(topic_orig_to_curated)}')
for k in list(topic_orig_to_curated.keys())[:5]:
    print(f'  ({k[0]}, "{k[1]}") → AI: "{topic_orig_to_curated[k][0]}", HC: "{topic_orig_to_curated[k][1]}"')

In [ ]:
# ── Image-based phrase matching by VisType ─────────────────────────────────────
import json

CANONICAL_VISTYPES = {'Area', 'Bar', 'Cont.-ColorPatn', 'Glyph', 'Grid', 'Line', 'Node-link', 'Point', 'Text'}

# Collect per-image records: (image_name, VisType, topic, original) → deduplicated
seen = set()
records = []
unmatched_phrases = set()

for idx, row in df_pairs.iterrows():
    # Process both sides: MoreComplex and LessComplex
    for img_col, vt_col, json_col_idx in [
        ('moreComplexImageName', 'VisTypesMoreComplexImageType', 18),
        ('lessComplexImageName', 'VisTypesLessComplexImageType', 24),
    ]:
        img_name = row.get(img_col)
        raw_vt = row.get(vt_col)
        if not isinstance(img_name, str) or not isinstance(raw_vt, str):
            continue
        raw_vt = raw_vt.strip()

        # Only keep exact canonical VisTypes
        if raw_vt not in CANONICAL_VISTYPES:
            continue

        # Parse JSON phrases
        json_str = df_pairs.iloc[df_pairs.index.get_loc(idx), json_col_idx]
        if not isinstance(json_str, str) or json_str.strip() == '':
            continue
        try:
            phrase_dict = json.loads(json_str)
        except json.JSONDecodeError:
            continue

        for topic, phrases_list in phrase_dict.items():
            if not isinstance(phrases_list, list):
                continue
            for orig_phrase in phrases_list:
                if not isinstance(orig_phrase, str) or orig_phrase.strip() == '':
                    continue
                orig_phrase = orig_phrase.strip()

                # Deduplicate: same image may appear in multiple rows
                dedup_key = (img_name, raw_vt, topic, orig_phrase)
                if dedup_key in seen:
                    continue
                seen.add(dedup_key)

                ai, hc = topic_orig_to_curated.get((topic, orig_phrase), ('', ''))
                if not ai and not hc:
                    unmatched_phrases.add(orig_phrase)
                records.append({
                    'ImageName': img_name,
                    'VisType': raw_vt,
                    'Topic': topic,
                    'Original': orig_phrase,
                    'AIcurated': ai,
                    'HumanCurated': hc,
                })

df_image_phrases = pd.DataFrame(records)
csv_path = 'data_processing/image_phrase_matching.csv'
df_image_phrases.to_csv(csv_path, index=False)
n_images = df_image_phrases['ImageName'].nunique()
print(f'Records: {len(df_image_phrases)} ({n_images} unique images)')
print(f'Unique VisTypes: {sorted(df_image_phrases["VisType"].unique())}')
print(f'Unmatched phrases: {len(unmatched_phrases)}')
if unmatched_phrases:
    for p in sorted(unmatched_phrases):
        print(f'  - "{p}"')
print(f'\nSaved {csv_path}')
df_image_phrases.head(10)

In [ ]:
# ── Per-topic unique words with image counts (Original, AIcurated, HumanCurated) ─
frames = []
for topic in topic_cols:
    subset = df_image_phrases[df_image_phrases['Topic'] == topic]
    col_data = {}
    for src_col, label in [('Original', 'Original'), ('AIcurated', 'AIcurated'), ('HumanCurated', 'HumanCurated')]:
        # Count images per lemma
        lemma_images = Counter()
        for img_name, grp in subset.groupby('ImageName'):
            phrases = grp[src_col].dropna().unique()
            img_lemmas = set()
            for phrase in phrases:
                img_lemmas |= extract_lemmas_from_phrase(phrase)
            for lemma in img_lemmas:
                lemma_images[lemma] += 1

        sorted_items = sorted(lemma_images.items())
        col_data[f'{topic}\n{label}'] = [w for w, _ in sorted_items]
        col_data[f'{topic}\n{label} Count'] = [str(c) for _, c in sorted_items]

    # Pad to same length within this topic
    if col_data:
        ml = max(len(v) for v in col_data.values())
        for k, v in col_data.items():
            col_data[k] = v + [''] * (ml - len(v))
    frames.append(pd.DataFrame(col_data))

    n_orig = len([x for x in col_data[f'{topic}\nOriginal'] if x])
    n_ai   = len([x for x in col_data[f'{topic}\nAIcurated'] if x])
    n_hc   = len([x for x in col_data[f'{topic}\nHumanCurated'] if x])
    print(f'{topic}: Orig={n_orig}, AI={n_ai}, HC={n_hc}')

global_max = max(len(f) for f in frames)
frames = [f.reindex(range(global_max)).fillna('') for f in frames]
df_topics = pd.concat(frames, axis=1)

csv_path = 'data_processing/unique_words_by_topic.csv'
df_topics.to_csv(csv_path, index=False)
print(f'\nSaved {csv_path}  ({len(df_topics)} rows, {len(df_topics.columns)} columns)')
df_topics.head(10)

In [ ]:
# ── Unique words by VisType with image counts (Original, AIcurated, HumanCurated)
CANONICAL_VISTYPES = ['Area', 'Bar', 'Cont.-ColorPatn', 'Glyph', 'Grid', 'Line', 'Node-link', 'Point', 'Text']

frames = []
for vt in CANONICAL_VISTYPES:
    subset = df_image_phrases[df_image_phrases['VisType'] == vt]
    col_data = {}
    for src_col, label in [('Original', 'Original'), ('AIcurated', 'AIcurated'), ('HumanCurated', 'HumanCurated')]:
        # Count images per lemma
        lemma_images = Counter()
        for img_name, grp in subset.groupby('ImageName'):
            phrases = grp[src_col].dropna().unique()
            img_lemmas = set()
            for phrase in phrases:
                img_lemmas |= extract_lemmas_from_phrase(phrase)
            for lemma in img_lemmas:
                lemma_images[lemma] += 1

        sorted_items = sorted(lemma_images.items())
        col_data[f'{vt}\n{label}'] = [w for w, _ in sorted_items]
        col_data[f'{vt}\n{label} Count'] = [str(c) for _, c in sorted_items]

    # Pad to same length within this VisType
    if col_data:
        ml = max(len(v) for v in col_data.values())
        for k, v in col_data.items():
            col_data[k] = v + [''] * (ml - len(v))

    frames.append(pd.DataFrame(col_data))
    n_orig = len([x for x in col_data[f'{vt}\nOriginal'] if x])
    n_ai   = len([x for x in col_data[f'{vt}\nAIcurated'] if x])
    n_hc   = len([x for x in col_data[f'{vt}\nHumanCurated'] if x])
    print(f'{vt}: Orig={n_orig}, AI={n_ai}, HC={n_hc}')

global_max = max(len(f) for f in frames)
frames = [f.reindex(range(global_max)).fillna('') for f in frames]
df_vistype = pd.concat(frames, axis=1)

csv_path = 'data_processing/unique_words_by_vistype.csv'
df_vistype.to_csv(csv_path, index=False)
print(f'\nSaved {csv_path}  ({len(df_vistype)} rows, {len(df_vistype.columns)} columns)')
df_vistype.head(10)